# 10. Memory Profiling & In-Place Optimization: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **10. Memory Profiling & In-Place Optimization**. Large-scale numerical workflows can easily trigger memory exhaustion if intermediate arrays are repeatedly allocated during operations like `A = A + B`. This notebook explores in-place arithmetic operators (`+=`, `*=`), pre-allocated destination buffers using the `out=` parameter across ufuncs, and memory footprint introspection comparing `sys.getsizeof()` against `.nbytes`.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 In-Place Mutation with `+=` and `*=`
- [x] 🔹 Pre-Allocated Buffers with `out=`
- [x] 🔹 Memory Profiling: `sys.getsizeof()` vs `.nbytes`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14251 clean aligned rows):
- amounts array: shape (14251,), dtype float64
- fraud_flags array: shape (14251,), dtype int8
- account_ages array: shape (14251,), dtype float32


### 🔹 In-Place Mutation with `+=` and `*=`
- **What it does:** Scales numerical values in-place without heap allocations.
- **Syntax:** `+=`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies In-Place Mutation with `+=` and `*=` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [2]:
amt_buf = amounts[:1000].copy()
orig_id = id(amt_buf)
amt_buf *= 1.05  # In-place
print('Memory address preserved after in-place mutation?:', id(amt_buf) == orig_id)

Memory address preserved after in-place mutation?: True


### 🔹 Pre-Allocated Buffers with `out=`
- **What it does:** Directs addition of fee buffers into a pre-allocated destination.
- **Syntax:** `out=`
  - **Parameters:**
    - `row_indexer` (*scalar, slice, list, or boolean mask*): Row identifier(s).
  - **Optional Parameters:**
    - `col_indexer` (*scalar, slice, list, or boolean mask*): Column identifier(s).
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Pre-Allocated Buffers with `out=` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [3]:
dest_buffer = np.empty_like(amounts[:1000])
np.add(amounts[:1000], 2.50, out=dest_buffer)
print('Output Buffer Result (first 5):', dest_buffer[:5].round(2))

Output Buffer Result (first 5): [ 610.28 1821.61   66.58 1028.23  775.24]


### 🔹 Memory Profiling: `sys.getsizeof()` vs `.nbytes`
- **What it does:** Returns the total number of elements contained across all dimensions.
- **Syntax:** `ndarray.size / Series.size`
  - **Parameters:**
    - `key` (*hashable*): The key to look up in the dictionary.
  - **Optional Parameters:**
    - `default` (*object, default None*): Fallback value returned if key is missing.
- **Key Note:** For 2D array of shape (4, 4), `.size == 16`.
- **Dataset Application & Code Demonstration:** Applies Memory Profiling across the extracted numeric transaction `amounts` array to compute performance metrics.


In [4]:
print(f'Raw Binary Buffer: {amounts.nbytes / 1024:.2f} KB')
print(f'Wrapper Object Overhead: {sys.getsizeof(amounts)} bytes')

Raw Binary Buffer: 111.34 KB
Wrapper Object Overhead: 112 bytes


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Zero-Allocation Fee & Tax Formula Pipeline
- **Objective:** Q1: Zero-Allocation Fee & Tax Formula Pipeline
- **Approach:** Compute `final_amount = (amount * 1.02) + 0.30` without allocating any intermediate temporary arrays.
- **Syntax:** `np.multiply(amounts, 1.02, out=res); np.add(res, 0.30, out=res)`

In [5]:
res = np.empty(1000, dtype=np.float64)
np.multiply(amounts[:1000], 1.02, out=res)
np.add(res, 0.30, out=res)
print('Zero-Allocation Pipeline Result Head:', res[:5].round(2))

Zero-Allocation Pipeline Result Head: [ 620.24 1855.79   65.66 1046.54  788.49]
